In [1]:
import torch
from transformers import BertTokenizer, BertModel
import umap.umap_ as umap
import plotly.express as px
import pandas as pd
import numpy as np

In [2]:
corpus = {
    #  Quasi-Monosemia (Bassa complessità)
    "cat (Animale)": [
        "My cat loves to sleep on the sofa.",
        "The stray cat caught a mouse in the barn.",
        "She adopted a beautiful rescue cat."
    ],

    # Omonimia Netta (Separazione netta)
    "bass (Strumento)": [
        "He plays the bass guitar in a rock band.",
        "The bass was turned up so loud the speakers shook."
    ],
    "bass (Pesce)": [
        "We caught a large sea bass on our fishing trip.",
        "Freshwater bass is delicious when grilled."
    ],

    # Polisemia
    "bank (Finanza)": [
        "I deposited my money in the bank today.",
        "The central bank raised interest rates."
    ],
    "bank (Fiume)": [
        "The boat was moored on the muddy bank.",
        "He sat on the river bank and watched the water."
    ],

    # Polisemia Regolare (Grinding: Animale -> Carne)
    "chicken (Animale)": [
        "The farmer feeds the chicken every morning.",
        "A fox sneaked in and stole a chicken."
    ],
    "chicken (Carne)": [
        "We are having roast chicken for dinner.",
        "This chicken soup is very comforting."
    ],

    # Alta Complessità / Continuum Semantico
    "run (Movimento)": [
        "I go for a run in the park every morning.",
        "He had to run quickly to catch the bus."
    ],
    "run (Azienda)": [
        "She knows how to run a successful business.",
        "It is difficult to run a large multinational corporation."
    ],
    "run (Informatica)": [
        "The program will run automatically on startup.",
        "I need to run a virus scan on my computer."
    ]
}

# Dizionario per far capire allo script quale parola estrarre per ogni categoria
target_words = {
    "cat (Animale)": "cat",
    "bass (Strumento)": "bass", "bass (Pesce)": "bass",
    "bank (Finanza)": "bank", "bank (Fiume)": "bank",
    "chicken (Animale)": "chicken", "chicken (Carne)": "chicken",
    "run (Movimento)": "run", "run (Azienda)": "run", "run (Informatica)": "run"
}

In [3]:
#Inizializzazione BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [4]:
def get_all_layers_embeddings(sentence, target):
    #Estraiamo gli embeddings per ogni layyer
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        # outputs.hidden_states è una tupla di 13 tensori (Layer 0 = input statico, Layer 1-12 = transformer)
        all_hidden_states = outputs.hidden_states

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    try:
        target_idx = tokens.index(target.lower())
        # Raccogliamo un vettore per ogni layer (da 0 a 12)
        layer_vectors = []
        for layer_idx, hidden_state in enumerate(all_hidden_states):
            vector = hidden_state[0][target_idx].numpy()
            layer_vectors.append((layer_idx, vector))
        return layer_vectors
    except ValueError:
        return None

In [5]:
#Estrazione ed elaborazione vettoriale per tutti i layer
data = []
for label, sentences in corpus.items():
    target = target_words[label]
    for sent in sentences:
        layers_data = get_all_layers_embeddings(sent, target)
        if layers_data:
            for layer_idx, emb in layers_data:
                data.append({
                    'Categoria': label,
                    'Parola': target,
                    'Frase': sent,
                    'Layer': layer_idx,
                    'embedding': emb
                })

df_data = pd.DataFrame(data)
embeddings_matrix = np.stack(df_data['embedding'].values)

In [6]:
#Riduzione Dimensionale Globale
#Considerando la mole di dati, questo è il punto in cui il codice può metterci piu tempo di elaborazione
# Abbassiamo min_dist per far raggruppare meglio i cluster finali
reducer = umap.UMAP(n_neighbors=4, min_dist=0.2, random_state=42)
projections = reducer.fit_transform(embeddings_matrix)
df_data['UMAP_X'] = projections[:, 0]
df_data['UMAP_Y'] = projections[:, 1]
# Ordiniamo il dataframe per layer
df_data = df_data.sort_values(by=['Frase', 'Layer'])

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
fig = px.line_3d(
    df_data,
    x='UMAP_X', y='UMAP_Y', z='Layer',
    color='Categoria',
    line_group='Frase',
    markers=True,
    hover_name='Categoria',
    hover_data={'Frase': True, 'Layer': True, 'UMAP_X': False, 'UMAP_Y': False, 'Categoria': False},
    title="La Cascata della Polisemia: Evoluzione Semantica nei Layer di BERT"
)

# Ottimizzazione estetica per la presentazione
fig.update_traces(line=dict(width=5), marker=dict(size=5, opacity=0.8))
fig.update_layout(
    scene=dict(
        xaxis_title='Spazio Semantico (X)',
        yaxis_title='Spazio Semantico (Y)',
        zaxis_title='Profondità Rete (Layer 0-12)',
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.6))
    ),
    legend_title_text='Senso Analizzato',
    margin=dict(l=0, r=0, b=0, t=40)
)

In [8]:
#html_filename = "Progetto_Polisemia_BERT.html"
#fig.write_html(html_filename)

# Analisi dei risultati e spiegazione del grafico

### Layer 0:
Questo è lo stato di ingresso, dove le reti non hanno ancora letto il contesto. Per esempio possiamo vedere come i pallini della parola *bank* siano sovrapposti, anche se si parla di fiumi o tassi di interesse. Questo imita esattamente i vecchi modelli Word2Vec.

### Layer 1-5:
Notiamo che i punti iniziano a muoversi verso direzioni diverse in base al contesto della frase

### Layer 6-11:
*  I punti con significati simili si muoveranno leggermente, però resteranno evidenti come cluster unico, dimostrando una varianza all'interno del cluster bassa.
(parola esempio: cat)

* I punti con omonimia netta si divideranno creando isole separate e distanti (parola esempio :    bass --> pesce vs strumento musicale)

* I punti con alta entropia si spargeranno formando gradienti senza confini netti, confermando l'alta separabilità sfumata.

Tramite queste visualizzazioni abbiamo mostrato visivamente il momento di disambiguazione

### Layer 12:
Rappresentazione finale delle decisioni del modello, dove i sensi sono completamente formati in cluster vettoriali distinti

Possiamo vedere come frasi con parole come "cat" salgono verticalmente senza movimenti improvvisi, mostrando la bassa entropia e l'assenza di ambiguità in quella parola.



